In [48]:
"""
Constructs the DFT of the dihedral group in SageMath using knowledge of the representation theory.
""";

In [49]:
#construct dihedral group of order n over GF(q)
n = 3; print("n =", n)
var('z')
#forget()
#assume(z**n == 1)
#omega = z; print("omega =", omega)
K.<z> = CyclotomicField(n)
omega = z; print("omega =", omega)
G = DihedralGroup(n); print("G =", G)
#gens = G.gens()
r = [g for g in G if g.order() == n][0]
s = [g for g in G if g.order() == 2 and g != r**(n//2)][0]
print("r =", r)
print("s =", s)

n = 3
omega = z
G = Dihedral group of order 6 as a permutation group
r = (1,3,2)
s = (2,3)


In [50]:
# returns (0, k) if g = r^k and (1, k) if g = s*r^k
def express_in_gens(g):
    for k in range(n):
        if g == r**k:
            return (0, k)
    for k in range(n):
        if g == s * r**k:
            return (1, k)

In [51]:
# n odd, we have two 1-dim'l irreps and (n-1)/2 2-dim'l irreps
# the 1-dim's irreps are trivial and sign
# the 2-dim'l irreps are given by rotation matrices and a flip matrix
def rho_odd(k, g):
    (s_exp, r_exp) = express_in_gens(g)
    if k == 0:
        return matrix([1])
    if k == -1:
        if s_exp == 0:
            return matrix([1])
        if s_exp == 1:
            return matrix([-1])
    if k >= 1:
        if s_exp == 0:
            return matrix([[omega**(k*r_exp), 0], [0, omega**(-k*r_exp)]])
        if s_exp == 1:
            return matrix([[0, omega**(k*r_exp)], [omega**(-k*r_exp), 0]])

In [52]:
def dft_matrix_odd():
    assert n % 2 == 1
    rows = []
    for g in G:
        row = [rho_odd(k, g).list() for k in range(-1,(n-1)//2 + 1)]
        rows.append(sum(row, []))
    return matrix(rows)

In [53]:
# for n even case
def rho_even(k, g):
    (s_exp, r_exp) = express_in_gens(g)
    if k == 0:   # trivial
        return matrix([1])
    if k == -1:  # sign of rotation
        return matrix([(-1)**r_exp])
    if k == -2:  # sign of reflection
        return matrix([(-1)**s_exp])
    if k == -3:  # total sign
        return matrix([(-1)**(r_exp + s_exp)])
    if k >= 1:
        if s_exp == 0:
            return matrix([[omega**(k*r_exp), 0], [0, omega**(-k*r_exp)]])
        if s_exp == 1:
            return matrix([[0, omega**(k*r_exp)], [omega**(-k*r_exp), 0]])

In [54]:
# form the DFT matrix for n even
def dft_matrix_even():
    assert n % 2 == 0
    rows = []
    for g in G:
        row = [rho_even(k, g).list() for k in range(-3, n//2)]
        rows.append(sum(row, []))
    return matrix(rows)

In [55]:
DFT_matrix = dft_matrix_odd() if n % 2 == 1 else dft_matrix_even(); print(DFT_matrix)

[     1      1      1      0      0      1]
[     1      1      z      0      0 -z - 1]
[     1      1 -z - 1      0      0      z]
[    -1      1      0      1      1      0]
[    -1      1      0      z -z - 1      0]
[    -1      1      0 -z - 1      z      0]


In [56]:
DFT_matrix.trace()

-2*z + 1

In [57]:
DFT_matrix.charpoly()

x^6 + (2*z - 1)*x^5 + (-6*z - 3)*x^4 + (5*z + 1)*x^3 + (-24*z - 9)*x^2 + (36*z + 36)*x - 54

In [58]:
DFT_matrix.eigenvalues()

[-1.456561162027096? - 1.144079954846116?*I,
 -0.7156434361425247? - 1.632899576656306?*I,
 -0.6885640009872164? + 1.852412312366662?*I,
 0.8723858562188725? - 1.739925111793133?*I,
 1.682952010437964? + 0.7377510605184450?*I,
 2.305430732500001? + 0.1946904628415694?*I]

In [59]:
# can normalize by 1/sqrt(D) to get a unitary matrix
print(DFT_matrix.conjugate_transpose() * DFT_matrix)

[6 0 0 0 0 0]
[0 6 0 0 0 0]
[0 0 3 0 0 0]
[0 0 0 3 0 0]
[0 0 0 0 3 0]
[0 0 0 0 0 3]


In [60]:
# form norm polynomial by acting on coefficients of characteristic polynomial by the Galois group
def norm_poly(f, n):

    def sigma(i):
        phi = K.hom([z**i])
        return f.map_coefficients(phi)
    
    units = [i for i in range(1, n) if gcd(i, n) == 1]  # (Z/nZ)^×
    result = prod(sigma(i) for i in units)
    return result.change_ring(QQ)

In [61]:
f = DFT_matrix.charpoly(); f

x^6 + (2*z - 1)*x^5 + (-6*z - 3)*x^4 + (5*z + 1)*x^3 + (-24*z - 9)*x^2 + (36*z + 36)*x - 54

In [62]:
Nf = norm_poly(f, n); Nf

x^12 - 4*x^11 + 7*x^10 - 21*x^9 + 54*x^8 - 93*x^7 + 165*x^6 - 297*x^5 + 657*x^4 - 1026*x^3 + 972*x^2 - 1944*x + 2916

In [63]:
Nf.is_irreducible()

True

In [64]:
Nf.galois_group()

Transitive group number 299 of degree 12

In [ ]:
G = TransitiveGroup(12, 299)
print(G.order())
print(G.is_solvable())
print(G.is_primitive())
print(G.structure_description())

1036800
False
False
(A6 x A6) : D4
